# Wavebit Tutorial 2
## Gradient and Optimization with [PyTorch](https://pytorch.org/)

In [1]:
import torch

import wavebit
# from wavebit.on_torch.optimizer.loss import LossDeflector

In [2]:
backend = 2  # Torch

tetm = 'te'  # 'te' or 'tm' — a declaration, so this takes the fast 1D path

top = 1.0  # superstrate: relative permittivity. 1.0 is vacuum
bot = 1.0  # substrate

theta = 0 * torch.pi / 180  # angle of incidence
# phi = 0 * torch.pi / 180  # angle of rotation

wavelength = 900

thickness = torch.tensor([500., 1000.])  # thickness of each layer, from top to bottom.
period = torch.tensor([1000.])  # length of the unit cell. Here it's 1D.

max_order = [10]

type_complex = torch.complex128
device = torch.device('cpu')

In [3]:
structure_1d_m = torch.tensor([
    [[0, 0, 0, 1, 1, 1, 1, 0, 0, 0, ]],
    [[1, 1, 1, 1, 0, 1, 1, 1, 1, 1, ]],
    ]) * 24 + 1.  # relative permittivity: 1 (vacuum) and 25

# The design variable is the permittivity itself, one number per pixel -- which is what a
# raster is for: a sampled field rather than an assignment, so every pixel is a knob. The
# gradient below is d(loss)/d(eps). It was d(loss)/d(n) before the material convention
# changed, and the two differ by the chain factor 2n, so old numbers do not carry over.

## 2.1 Gradient
Gradient can be calculated with the help of `torch.autograd` function.
Read this for further information: [A GENTLE INTRODUCTION TO TORCH.AUTOGRAD](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)

Gradient can be utilized to solve optimization problems. Here are examples that show couple of ways to get gradient or optimized values with or without predefined functions of wavebit.

### 2.1.1 Examples

Example 1: manually get gradient


In [ ]:
solver = wavebit.rcwa(backend=backend, tetm=tetm, top=top, bot=bot, theta=theta,
                      max_order=max_order, wavelength=wavelength, period=period, structure=structure_1d_m, 
                      thickness=thickness, type_complex=type_complex, device=device)

solver.structure.requires_grad = True
solver.thickness.requires_grad = True

result = solver.solve()
res = result.given
R, T = res.R, res.T
loss = T[T.shape[0] // 2, T.shape[1] // 2 + 1]

loss.backward()
print('structure gradient:')
print(solver.structure.grad)
print('thickness gradient:')
print(solver.thickness.grad)

Example 2: using predefined 'grad' function in Wavebit

In [5]:
class Loss:
    def __call__(self, result, *args, **kwargs):
        given, res_te, res_ti = result.given, result.s_inc, result.p_inc
        T = given.T
        center = [a // 2 for a in T.shape]
        res = T[center[0], center[1]+1]

        return res


loss_fn = Loss()

In [ ]:
solver = wavebit.rcwa(backend=backend, tetm=tetm, top=top, bot=bot, theta=theta,
                      max_order=max_order, wavelength=wavelength, period=period, structure=structure_1d_m, 
                      thickness=thickness, type_complex=type_complex, device=device)

pois = ['structure', 'thickness']  # Parameter Of Interests

forward = solver.solve

grad = solver.grad(pois, forward, loss_fn)
print('structure gradient:')
print(grad['structure'])
print('thickness gradient:')
print(grad['thickness'])

## 2.2 Optimization

### 2.2.1 Examples

Example 1

In [ ]:
solver = wavebit.rcwa(backend=backend, tetm=tetm, top=top, bot=bot, theta=theta,
                      max_order=max_order, wavelength=wavelength, period=period, structure=structure_1d_m, 
                      thickness=thickness, type_complex=type_complex, device=device)

solver.structure.requires_grad = True
solver.thickness.requires_grad = True
opt = torch.optim.SGD([solver.structure, solver.thickness], lr=1E-2, momentum=0.9)

for _ in range(3):
    result = solver.solve()
    res = result.given
    R, T = res.R, res.T

    center = T.shape[1] // 2
    loss = T[0, center + 1]

    loss.backward()
    opt.step()
    opt.zero_grad()

print('structure final:')
print(solver.structure)
print('thickness final:')
print(solver.thickness)

Example 2

In [ ]:
solver = wavebit.rcwa(backend=backend, tetm=tetm, top=top, bot=bot, theta=theta,
                     max_order=max_order, wavelength=wavelength, period=period, structure=structure_1d_m, 
                     thickness=thickness, type_complex=type_complex, device=device)


def forward_fn():

    result = solver.solve()
    res = result.given
    R, T = res.R, res.T

    center = T.shape[1] // 2
    loss = T[0, center + 1]
    return loss

pois = ['structure', 'thickness']
forward = forward_fn
loss_fn = lambda x: x
opt_torch = torch.optim.SGD
opt_options = {'lr': 1E-2,
               'momentum': 0.9,
               }

res = solver.fit(pois, forward, loss_fn, opt_torch, opt_options, iteration=3)

print('structure final:')
print(res[0])
print('thickness final:')
print(res[1])

Example 3

In [ ]:
solver = wavebit.rcwa(backend=backend, tetm=tetm, top=top, bot=bot, theta=theta, 
                     max_order=max_order, wavelength=wavelength, period=period, structure=structure_1d_m, 
                     thickness=thickness, type_complex=type_complex, device=device)

pois = ['structure', 'thickness']

forward = solver.solve
loss_fn = Loss()
opt_torch = torch.optim.SGD
opt_options = {'lr': 1E-2,
               'momentum': 0.9,
               }

res = solver.fit(pois, forward, loss_fn, opt_torch, opt_options, iteration=3)

print('structure final:')
print(res[0])
print('thickness final:')
print(res[1])